# AFM Force-Curve Analysis: INVOLS Determination

## Step-by-step Data Reduction

This notebook walks through the complete pipeline for extracting the
**Inverse Optical Lever Sensitivity (INVOLS)** from a hard-surface
AFM force curve.

**Goal:** convert the raw photodetector voltage into a physical
cantilever deflection (nm), which is the prerequisite for any
quantitative force measurement.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project modules (same directory or on PYTHONPATH)
from io_utils import FCConfig, read_lvm_raw, detect_turnaround, split_at_turnaround
from contact_point import (estimate_contact_approach, estimate_contact_retract,
                           fit_contact_region, compute_phase_angle)

# Display settings
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.labelsize': 11,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 2. Load the Raw Data

Each `.lvm` file contains four concatenated blocks of single-column voltage data:

| Rows | Content |
|------|---------|
| 1 … N | Approach deflection (V) |
| N+1 … 2N | Retract deflection (V) |
| 2N+1 … 3N | Approach Z-stage (V) |
| 3N+1 … 4N | Retract Z-stage (V) |

The parameter `N` = `num_app` is read from `config.txt`.

In [ ]:
# ── Adjust these paths to your data ──
lvm_path    = Path("path/to/ForceCurve/ForceCurve_00009.lvm")
config_path = Path("path/to/config.txt")

# Parse measurement config
config = FCConfig.from_file(config_path)
print(f"num_app = {config.num_app},  num_ret = {config.num_ret}")
print(f"Z-stage scale = {config.z_scale} µm/V")

# Read and concatenate approach + retract into continuous signals
raw = read_lvm_raw(lvm_path, num_app=config.num_app, num_ret=config.num_ret)
print(f"Total points: {len(raw.z_V)}")

## 3. Approach / Retract Separation — Turnaround Detection

The file-level split at `num_app` does **not** always coincide with the
physical turnaround of the Z-stage piezo.  Instead we find the actual
peak of the Z-stage signal and split there.

This ensures approach and retract curves **meet at maximum indentation**.

In [ ]:
turnaround = detect_turnaround(raw)
print(f"Turnaround at index {turnaround}")
print(f"  Z = {raw.z_V[turnaround]:.5f} V")
print(f"  Deflection = {raw.defl_V[turnaround]:.5f} V")

# Split at the true turnaround
fc = split_at_turnaround(raw, turnaround)

print(f"\nApproach: {len(fc.app_z_V)} points")
print(f"  Z: {fc.app_z_V[0]:.4f} → {fc.app_z_V[-1]:.4f} V")
print(f"Retract:  {len(fc.ret_z_V)} points")
print(f"  Z: {fc.ret_z_V[0]:.4f} → {fc.ret_z_V[-1]:.4f} V")

# Verify connection
print(f"\nConnection: app end Z == ret start Z? "
      f"{fc.app_z_V[-1] == fc.ret_z_V[0]}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(fc.app_z_V, fc.app_defl_V, lw=0.7, color='#0077bb', label='Approach')
ax.plot(fc.ret_z_V, fc.ret_defl_V, lw=0.7, color='#cc3311', label='Retract')
ax.plot(fc.app_z_V[-1], fc.app_defl_V[-1], 'Dk', ms=7, label='Turnaround')
ax.set_xlabel('Z-stage (V)'); ax.set_ylabel('Deflection (V)')
ax.set_title('Step 3: Full Force Curve after Turnaround Split')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Contact-Point Detection

We use a **threshold-free piecewise-linear fit**:

1. For each candidate split index *i*, fit an independent line to
   `[0, i)` (baseline) and `[i, N)` (contact).
2. Compute the total sum-of-squared residuals: `R(i) = R_base(i) + R_contact(i)`.
3. The contact point is the index that **minimises** R(i).

A coarse sweep (150 candidates) finds the region; a fine sweep (80) refines it.
No threshold parameter is needed.

- **Approach:** search 50 %–99.5 % (baseline early, contact late)
- **Retract:** search 0.5 %–50 % (contact early, baseline late)

In [ ]:
cp_app = estimate_contact_approach(fc.app_z_V, fc.app_defl_V)
cp_ret = estimate_contact_retract(fc.ret_z_V, fc.ret_defl_V)

print(f"Approach CP:  idx = {cp_app.index},  Z = {cp_app.z_V:.5f} V,"
      f"  defl = {cp_app.defl_V:.5f} V")
print(f"Retract  CP:  idx = {cp_ret.index},  Z = {cp_ret.z_V:.5f} V,"
      f"  defl = {cp_ret.defl_V:.5f} V")

# Distance between CPs (in nm)
z_scale = config.z_scale
cp_dist_nm = abs(cp_app.z_V - fc.ret_z_V[cp_ret.index]) * z_scale * 1000
print(f"\nCP distance: {cp_dist_nm:.1f} nm")

In [ ]:
fig, ax = plt.subplots()
ax.plot(fc.app_z_V, fc.app_defl_V, lw=0.7, color='#0077bb', label='Approach')
ax.plot(fc.ret_z_V, fc.ret_defl_V, lw=0.7, color='#cc3311', label='Retract')
ax.plot(cp_app.z_V, cp_app.defl_V, 'o', ms=9, mfc='none', mew=2,
        color='#0077bb', label=f'CP app (idx {cp_app.index})')
ax.plot(fc.ret_z_V[cp_ret.index], fc.ret_defl_V[cp_ret.index], 'o',
        ms=9, mfc='none', mew=2, color='#cc3311',
        label=f'CP ret (idx {cp_ret.index})')
ax.set_xlabel('Z-stage (V)'); ax.set_ylabel('Deflection (V)')
ax.set_title('Step 4: Contact Points')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Linear Fit of the Contact Region → INVOLS

On a **hard (infinitely stiff) surface**, every increment of Z-stage
displacement produces an equal increment of cantilever deflection.
The slope of `deflection (V)` vs `Z-stage (V)` in the contact region is:

$$\text{slope} = \frac{\Delta d \;(\text{V})}{\Delta Z \;(\text{V})}$$

Since $\Delta Z\,(\mu\text{m}) = \Delta Z\,(\text{V}) \times z_{\text{scale}}$
and on a hard surface $\Delta d\,(\text{nm}) = \Delta Z\,(\mu\text{m}) \times 1000$:

$$\text{INVOLS}\;(\text{nm/V}) = \frac{z_{\text{scale}} \times 1000}{\text{slope}}$$

We fit both contact regions independently.

In [ ]:
# Approach: from CP to end (turnaround)
fit_app = fit_contact_region(fc.app_z_V, fc.app_defl_V,
                             cp_app.index, len(fc.app_z_V),
                             z_scale=z_scale)

# Retract: from start (turnaround) to CP
fit_ret = fit_contact_region(fc.ret_z_V, fc.ret_defl_V,
                             0, cp_ret.index,
                             z_scale=z_scale)

print(f"Approach:  slope = {fit_app.slope:.2f} V/V,"
      f"  INVOLS = {fit_app.invols_nm_per_V:.1f} nm/V")
print(f"Retract:   slope = {fit_ret.slope:.2f} V/V,"
      f"  INVOLS = {fit_ret.invols_nm_per_V:.1f} nm/V")
print(f"\nAverage INVOLS = "
      f"{(fit_app.invols_nm_per_V + fit_ret.invols_nm_per_V)/2:.1f} nm/V")

phase = compute_phase_angle(fit_app.slope, fit_ret.slope)
print(f"Phase angle = {phase:.2f}°")

In [ ]:
fig, ax = plt.subplots()
a_z = fc.app_z_V[cp_app.index:]
a_d = fc.app_defl_V[cp_app.index:]
ax.scatter(a_z, a_d, s=2, color='#0077bb', label='Approach', zorder=2)
ax.plot(fit_app.x_fit, fit_app.y_fit, color='#004488', lw=2,
        label=f'Fit app (s={fit_app.slope:.1f})', zorder=3)
r_z = fc.ret_z_V[:cp_ret.index+1]
r_d = fc.ret_defl_V[:cp_ret.index+1]
ax.scatter(r_z, r_d, s=2, color='#cc3311', label='Retract', zorder=2)
ax.plot(fit_ret.x_fit, fit_ret.y_fit, color='#882211', lw=2,
        label=f'Fit ret (s={fit_ret.slope:.1f})', zorder=3)
ax.set_xlabel('Z-stage (V)'); ax.set_ylabel('Deflection (V)')
ax.set_title('Step 5: Contact Region with Linear Fits')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Force–Indentation Conversion

With INVOLS determined, we convert to physical units:

- **Deflection** (nm) = $d\,(\text{V}) \times \text{INVOLS}$
- **Indentation** $\delta = (Z - Z_\text{CP}) \times z_\text{scale} \times 1000 - (d - d_\text{CP}) \times \text{INVOLS}$
- **Force** $F = k \times \Delta d\,(\text{nm})$

The indentation $\delta$ accounts for cantilever bending: not all Z-stage
travel goes into indenting the sample — some deflects the cantilever.

In [ ]:
# Parameters
invols = (fit_app.invols_nm_per_V + fit_ret.invols_nm_per_V) / 2  # nm/V
k = 26.0  # N/m (AC160)

# Reference = approach CP
z_cp = fc.app_z_V[cp_app.index]
d_cp = fc.app_defl_V[cp_app.index]

# Approach: CP → turnaround
a_z = fc.app_z_V[cp_app.index:]
a_d = fc.app_defl_V[cp_app.index:]
a_delta = (a_z - z_cp) * z_scale * 1000 - (a_d - d_cp) * invols  # nm
a_F = k * (a_d - d_cp) * invols                                    # nN

# Retract: turnaround → CP
r_z = fc.ret_z_V[:cp_ret.index + 1]
r_d = fc.ret_defl_V[:cp_ret.index + 1]
r_delta = (r_z - z_cp) * z_scale * 1000 - (r_d - d_cp) * invols  # nm
r_F = k * (r_d - d_cp) * invols                                    # nN

print(f"Approach: δ = {a_delta[0]:.1f} → {a_delta[-1]:.1f} nm,"
      f"  F = {a_F[0]:.1f} → {a_F[-1]:.1f} nN")
print(f"Retract:  δ = {r_delta[0]:.1f} → {r_delta[-1]:.1f} nm,"
      f"  F = {r_F[0]:.1f} → {r_F[-1]:.1f} nN")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(a_delta, a_F, s=2, color='#0077bb', label='Approach', zorder=2)
ax.scatter(r_delta, r_F, s=2, color='#cc3311', label='Retract', zorder=2)
ax.axhline(0, lw=0.5, color='grey')
ax.axvline(0, lw=0.5, color='grey')
ax.set_xlabel('δ (nm)'); ax.set_ylabel('F (nN)')
ax.set_title('Step 6: Force vs Indentation')
ax.legend(); plt.tight_layout(); plt.show()

## 7. Why Do Approach and Retract INVOLS Differ?

On a perfectly rigid surface with an ideal instrument, both slopes
would be identical.  In practice several effects introduce a systematic
difference:

### 7.1 Piezo Hysteresis

The Z-stage piezo actuator has inherent voltage–displacement hysteresis.
When the voltage ramps up (approach) vs. down (retract), the *actual*
stage displacement per volt differs.  Since INVOLS uses a single nominal
`z_scale`, the two slopes map to different INVOLS values.

### 7.2 Viscous Drag

At finite approach/retract speeds, viscous drag from the surrounding
medium exerts a velocity-dependent force on the cantilever with
**opposite sign** for the two directions, shifting the apparent slope.

### 7.3 Photodetector Non-linearity

The PSD has a linear response only near its centre.  At the extremes
of the contact region sensitivity drops; because approach and retract
traverse the non-linear zone in opposite order, fitted slopes differ.

### 7.4 Adhesion and Capillary Forces

Retract curves often show adhesion near pull-off.  The attractive
interaction slightly steepens the retract slope relative to approach.

### Best Practice

**Average approach and retract INVOLS** to cancel first-order symmetric
errors.  If the discrepancy exceeds ~10 %, investigate piezo
non-linearity, excessive speed, or liquid drag.

## Summary

| Step | Operation | Result |
|------|-----------|--------|
| 1 | Read `.lvm` file | Raw V signals |
| 2 | Find turnaround (peak Z) | Proper approach/retract split |
| 3 | Piecewise-linear CP detection | Contact point indices |
| 4 | Linear fit of contact region | Slopes (V/V) |
| 5 | INVOLS = z_scale × 1000 / slope | nm/V conversion factor |
| 6 | δ, F computation | Physical force curve |

The Streamlit app (`app.py`) performs all these steps interactively
with real-time parameter adjustment.